# Rung 1/4/5 — standard-scale induction run and the head-validation cascade (GPU)

Issue #169. One Kaggle/Colab session, decided **before** seeing results:

| Step | What runs | Why |
|------|-----------|-----|
| 1 | `exp1 --scale large --seeds 0,1,2` (stateful: checkpoint every 500 epochs, resume) | The strongest single attempt at a confirmed induction head — the scaled config the closed exp7 PR intended (vocab 2048, seq 128, d_model 256, 4L/8H, 1500 epochs, fresh batches) |
| 2 | If step 1 found **no** head: `exp1 --standard --seeds 0,1,2` | The canonical MP-10 config Rungs 1/4/5 share — the issue's named fallback |
| 3 | `exp4 --scale large/--standard --seeds 0,1,2` | Activation patching + head ablation + path patching, gated internally on a detected head — the unit-tested-only path patching gets its first end-to-end run |
| 4 | `exp5 --activations-from <rolling checkpoint>` | The first SAE on real activations from a model with a *confirmed* head (the harvest fix landed 2026-10-12; rolling checkpoints now load) |

**Falsifiers, fixed in advance:**
- The detector is planted-tested (tests/test_planted_induction_detector.py) *before* this run — a "0 heads" here is a property of the model, not of untested detection code.
- If step 1 AND step 2 both find no head, that is the result: report it with manifests, do not tune until one appears.
- Every step writes `--manifest-path` under `results/`; nothing is edited by hand afterwards.

**Before running:** enable a GPU (Kaggle: Settings -> Accelerator -> GPU T4/P100, Internet On; Colab: Runtime -> Change runtime type -> GPU). Sessions are ~9h; every long step is resumable — re-run its cell after a disconnect.

In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifest's git_sha is a real commit
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
!pip install -q uv
!uv sync --frozen
!uv run python -c "import torch; assert torch.cuda.is_available(), 'no GPU — enable one before running'; print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# Step 1 — the scaled run (exp7's config, without exp7). Resumable: re-run
# this cell after a disconnect; each seed picks up its rolling checkpoint.
!uv run python -m src.experiments.exp1_induction_heads --scale large --seeds 0,1,2 --checkpoint-dir checkpoints/induction_large --checkpoint-every 500 --resume --manifest-path results/induction_scale_large.json

In [ ]:
# Which seeds detected an induction head? This decides the cascade.
import json

m = json.load(open("results/induction_scale_large.json"))
heads = []
for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
    n = int(metrics["total_induction_heads"])
    heads.append(n)
    print(f"seed {seed}: {n} induction head(s), val acc {metrics['final_val_acc']:.4f}, "
          f"peak diag+1 mass {metrics['peak_diag1_mass']:.3f}")
LARGE_FOUND_HEAD = any(n > 0 for n in heads)
print(f"\nSCALE=large found a head: {LARGE_FOUND_HEAD}")

## If step 1 found no head

Run the canonical standard-scale config below (the issue's named fallback). Skip it if a head was already found — more scale is not more information once the detector has fired.

In [ ]:
# Step 2 — canonical standard scale (only meaningful if step 1 found nothing).
if LARGE_FOUND_HEAD:
    print("A head was found at large scale — skipping the standard-scale fallback.")
else:
    !uv run python -m src.experiments.exp1_induction_heads --standard --fresh-batches --seeds 0,1,2 --checkpoint-dir checkpoints/induction_standard --checkpoint-every 250 --resume --manifest-path results/induction_standard.json

In [ ]:
# Step 3 — Rung 4 at the SAME config as the run that found the head.
# exp4 gates head ablation and path patching on detection internally: with no
# head they are skipped (and reported as 0.0), which is the honest outcome.
if LARGE_FOUND_HEAD:
    !uv run python -m src.experiments.exp4_circuit_patching --scale large --seeds 0,1,2 --manifest-path results/exp4_scale_large.json
else:
    !uv run python -m src.experiments.exp4_circuit_patching --standard --fresh-batches --seeds 0,1,2 --manifest-path results/exp4_standard.json

In [ ]:
# Step 4 — the SAE on real activations from the checkpoint of the first
# seed that detected a head. Rolling checkpoints load since the 2026-10-12
# harvest fix; --source-* must match how that model was trained.
import glob
import os

def first_head_seed():
    m = json.load(open("results/induction_scale_large.json")) if LARGE_FOUND_HEAD else json.load(open("results/induction_standard.json"))
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        if int(metrics["total_induction_heads"]) > 0:
            return seed
    return None

seed = first_head_seed()
if seed is None:
    print("No seed detected an induction head — the exp5 real-activation run has no"
          "confirmed-head checkpoint to harvest from. This is the honest outcome;"
          "the synthetic SAE baseline already reproduces.")
else:
    ckpt_dir = "checkpoints/induction_large" if LARGE_FOUND_HEAD else "checkpoints/induction_standard"
    ckpt = os.path.join(ckpt_dir, f"exp1_checkpoint_seed{seed}.pt")
    if LARGE_FOUND_HEAD:
        src = "--source-vocab-size 2048 --source-seq-len 128 --source-d-model 256 --source-n-layers 4 --source-n-heads 8"
        d_model = 256
    else:
        src = "--source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4"
        d_model = 64
    !uv run python -m src.experiments.exp5_sae_dashboard --activations-from $ckpt $src --d-model $d_model --seeds 42 --manifest-path results/exp5_real_activations.json

In [ ]:
# Collect every manifest this session produced, then download.
import glob

manifests = sorted(glob.glob("results/induction_*.json") + glob.glob("results/exp4_*.json") + glob.glob("results/exp5_real_activations.json"))
print("manifests:", manifests)
!zip -j induction_run_results.zip $manifests
try:
    from google.colab import files
    files.download("induction_run_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")

## After downloading

1. Unzip into a scratch dir, then pin: `python scripts/pin_remote_run.py --from <dir>` — it validates provenance (git sha must exist locally, clean tree, seeds/aggregate contract) and installs only what passes.
2. `make verify-claims` must stay green — new manifests need their `<!-- manifest: results/... -->` tags next to the numbers they back.
3. Report exactly what happened, including "no head at either scale": the answer to issue #169 is data, either way.
4. If a head WAS found: tell me to update the Rung 1 section of portfolio/RESULTS.md with the new numbers — the 0-head claim it carries is retracted by this run.